# SSIF_V3：CSV 轉換與資料驗證（繁體中文）

本 Notebook 僅處理：

1. 掛載 Google Drive 並同步最新版程式。
2. 檢查 `combined_data.csv` schema。
3. 使用真實前兩列做端到端轉換測試。
4. 完整 CSV scan、轉換與 archive validation。

模型訓練請使用獨立 Notebook：`notebooks/SSIF_V3_Model_Training_ZH_TW.ipynb`。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 安全同步 GitHub repository


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_ROOT = Path('/content/SSIF_V3')
REPO_URL = 'https://github.com/oceanicdayi/SSIF_V3.git'

def run_checked(command, cwd=None, capture=False):
    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=capture,
    )
    if capture:
        if result.stdout:
            print(result.stdout, end='')
        if result.stderr:
            print(result.stderr, end='')
    if result.returncode:
        raise RuntimeError(
            f"exit {result.returncode}: " + " ".join(map(str, command))
        )
    return result

# 先離開 repository，避免更新或重建目前工作目錄時出錯。
os.chdir('/content')

if (REPO_ROOT / '.git').is_dir():
    try:
        run_checked(['git', '-C', str(REPO_ROOT), 'fetch', '--prune', 'origin'])
        run_checked(['git', '-C', str(REPO_ROOT), 'reset', '--hard', 'origin/main'])
        run_checked(['git', '-C', str(REPO_ROOT), 'clean', '-fd'])
    except RuntimeError:
        os.chdir('/content')
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)])
else:
    os.chdir('/content')
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    run_checked(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)])

REPO_SHA = run_checked(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'],
    capture=True,
).stdout.strip()
print('Repository commit:', REPO_SHA)

_ = run_checked([
    'python', '-m', 'pip', 'install', '-q',
    '-r', str(REPO_ROOT / 'requirements.txt'),
])


## 2. 路徑、CSV 大欄位設定與執行開關


In [ ]:
import csv
import json
import sys

import pandas as pd
import torch
from IPython.display import display

# intensity、stids 等巢狀欄位可能遠大於 csv 預設的 131072 bytes。
_csv_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(_csv_limit)
        break
    except OverflowError:
        _csv_limit //= 10

assert csv.field_size_limit() > 131072

DATA_CSV = Path('/content/drive/MyDrive/00_SSIF/combined_data.csv')
WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')
TRAIN_DATA = WORK_ROOT / 'data' / 'training_archive_json'
REPORT_DIR = WORK_ROOT / 'reports'
SCAN_REPORT = REPORT_DIR / 'combined_data_scan.json'
VALIDATION_REPORT = REPORT_DIR / 'converted_archive_validation.json'

RUN_FULL_SCAN = False
RUN_FULL_CONVERSION = False
RUN_FULL_VALIDATION = False

# 完整轉換使用 --overwrite，會重建 TRAIN_DATA。必須另外明確開啟。
ALLOW_REBUILD_TRAIN_DATA = False
DUPLICATE_POLICY = 'skip-identical'

REPORT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DATA.parent.mkdir(parents=True, exist_ok=True)

assert DATA_CSV.is_file(), f'找不到 CSV：{DATA_CSV}'
print('CSV field limit:', csv.field_size_limit())
print('CSV size (GB):', round(DATA_CSV.stat().st_size / 1024**3, 3))
print('Existing converted events:', len(list(TRAIN_DATA.glob('event_*.json'))))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 3. Converter smoke test 與真實 CSV schema 檢查


In [ ]:
_ = run_checked(
    ['python', 'smoke_test_combined_csv_conversion.py'],
    cwd=REPO_ROOT,
)

inspect_result = run_checked(
    [
        'python', 'combined_csv_to_ssif_json.py', 'inspect',
        '--csv', str(DATA_CSV),
        '--rows', '2',
        '--horizon', '120',
    ],
    cwd=REPO_ROOT,
    capture=True,
)

inspect_summary = json.loads(inspect_result.stdout)
assert inspect_summary['layout'] == 'event_json'
assert {'eq_info', 'intensity'}.issubset(inspect_summary['columns'])
assert len(inspect_summary['sample_rows']) == 2

display(pd.DataFrame(inspect_summary['sample_rows']))
print('PASS: real CSV schema')


## 4. 真實前兩列端到端轉換測試


In [ ]:
SAMPLE_CSV = Path('/content/combined_data_sample_2rows.csv')
SAMPLE_OUT = Path('/content/ssif_combined_sample_events')
shutil.rmtree(SAMPLE_OUT, ignore_errors=True)

with DATA_CSV.open('r', encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    fieldnames = list(reader.fieldnames or [])
    rows = []
    for index, row in enumerate(reader):
        rows.append(row)
        if index == 1:
            break

assert len(rows) == 2
assert {'eq_info', 'intensity'}.issubset(fieldnames)

with SAMPLE_CSV.open('w', encoding='utf-8', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

_ = run_checked(
    [
        'python', 'combined_csv_to_ssif_json.py', 'convert',
        '--csv', str(SAMPLE_CSV),
        '--output-dir', str(SAMPLE_OUT),
        '--horizon', '120',
        '--duplicate-policy', 'skip-identical',
        '--error-policy', 'error',
        '--overwrite',
    ],
    cwd=REPO_ROOT,
)

_ = run_checked(
    [
        'python', 'combined_csv_to_ssif_json.py', 'validate',
        '--data-dir', str(SAMPLE_OUT),
        '--horizon', '120',
    ],
    cwd=REPO_ROOT,
)

sample_files = sorted(SAMPLE_OUT.glob('event_*.json'))
assert len(sample_files) == 2

sample_event = json.loads(sample_files[0].read_text(encoding='utf-8'))
assert isinstance(sample_event.get('eq_info'), dict)
assert isinstance(sample_event.get('intensity'), dict)
assert sample_event['intensity']

first_station = next(iter(sample_event['intensity']))
assert len(sample_event['intensity'][first_station]) == 120

print('PASS: real two-row conversion')
print('Example:', sample_files[0].name)
print('Stations:', len(sample_event['intensity']))


## 5. 完整 CSV scan

先將 `RUN_FULL_SCAN=True`，重新執行第 2 節與本節。Scan 只讀取 CSV，不會修改轉換後 archive。


In [ ]:
if RUN_FULL_SCAN:
    _ = run_checked(
        [
            'python', 'combined_csv_to_ssif_json.py', 'scan',
            '--csv', str(DATA_CSV),
            '--horizon', '120',
            '--max-errors', '100',
            '--report', str(SCAN_REPORT),
        ],
        cwd=REPO_ROOT,
    )
else:
    print('RUN_FULL_SCAN=False：未執行完整 scan')

scan_summary = None
if SCAN_REPORT.is_file():
    scan_summary = json.loads(SCAN_REPORT.read_text(encoding='utf-8'))
    display(pd.DataFrame.from_dict(
        scan_summary['counters'],
        orient='index',
        columns=['count'],
    ))
    if scan_summary['errors']:
        display(pd.DataFrame(scan_summary['errors']))
    if scan_summary['duplicate_examples']:
        display(pd.DataFrame(scan_summary['duplicate_examples']))

    print('Scan report:', SCAN_REPORT)


## 6. 完整 CSV 轉換

只有 scan 顯示 `row_errors == 0` 時才執行。完整轉換會重建 `TRAIN_DATA`，因此需同時設定：

```python
RUN_FULL_CONVERSION = True
ALLOW_REBUILD_TRAIN_DATA = True
```


In [ ]:
if RUN_FULL_CONVERSION:
    assert ALLOW_REBUILD_TRAIN_DATA, (
        '完整轉換會重建 TRAIN_DATA；請明確設定 ALLOW_REBUILD_TRAIN_DATA=True'
    )
    assert SCAN_REPORT.is_file(), '請先執行完整 scan'
    scan_summary = json.loads(SCAN_REPORT.read_text(encoding='utf-8'))
    counters = scan_summary['counters']

    assert counters.get('row_errors', 0) == 0, scan_summary['errors']
    if counters.get('identity_conflicts', 0):
        assert DUPLICATE_POLICY in {'merge', 'suffix'}, (
            '存在 identity_conflicts；請先審查 duplicate_examples'
        )

    process = subprocess.run(
        [
            'python', 'combined_csv_to_ssif_json.py', 'convert',
            '--csv', str(DATA_CSV),
            '--output-dir', str(TRAIN_DATA),
            '--horizon', '120',
            '--duplicate-policy', DUPLICATE_POLICY,
            '--error-policy', 'error',
            '--max-errors', '100',
            '--overwrite',
        ],
        cwd=REPO_ROOT,
        text=True,
)

    if process.returncode:
        error_report = TRAIN_DATA / 'conversion_error_report.json'
        if error_report.is_file():
            print(error_report.read_text(encoding='utf-8'))
        raise RuntimeError('完整 CSV 轉換失敗')

    conversion_summary = json.loads(
        (TRAIN_DATA / 'conversion_summary.json').read_text(encoding='utf-8')
    )
    print(json.dumps(conversion_summary, ensure_ascii=False, indent=2))
else:
    print('RUN_FULL_CONVERSION=False：保留現有轉換結果')


## 7. 驗證完整轉換 archive


In [ ]:
if RUN_FULL_VALIDATION:
    assert TRAIN_DATA.is_dir()
    assert any(TRAIN_DATA.glob('event_*.json'))

    _ = run_checked(
        [
            'python', 'combined_csv_to_ssif_json.py', 'validate',
            '--data-dir', str(TRAIN_DATA),
            '--horizon', '120',
            '--max-errors', '100',
            '--report', str(VALIDATION_REPORT),
        ],
        cwd=REPO_ROOT,
)

    validation = json.loads(
        VALIDATION_REPORT.read_text(encoding='utf-8')
    )
    display(pd.DataFrame.from_dict(
        validation['counters'],
        orient='index',
        columns=['count'],
    ))

    assert validation['counters'].get('errors', 0) == 0
    assert validation['counters'].get('event_json', 0) > 0
    print('PASS: full archive validation')
else:
    print('RUN_FULL_VALIDATION=False')


## 下一步：模型訓練

完成 archive validation 後，請改用：

- GitHub：`notebooks/SSIF_V3_Model_Training_ZH_TW.ipynb`
- Colab：`https://colab.research.google.com/github/oceanicdayi/SSIF_V3/blob/main/notebooks/SSIF_V3_Model_Training_ZH_TW.ipynb`

資料轉換與模型訓練分開，可避免重新執行轉換時誤刪模型或混淆研究版本。
